In [0]:
import json

from og_screening import SCREENING_VERSION, build_screen_delta, plan_screening


projects = [
    {"project_id": 1, "project_name": "North LNG Terminal", "country": "Brazil"},
    {"project_id": 2, "project_name": "East Refinery", "country": "India"},
]
opportunities = [
    {"opportunity_id": "s1", "opportunity_name": "North LNG Terminal", "country": "Brazil"},
    {"opportunity_id": "s2", "opportunity_name": "East Refinery", "country": "Chile"},
]
matched = [
    {"project_id": 1, "candidate_ids": ["s1"], "candidate_scores": [100.0]},
    {"project_id": 2, "candidate_ids": [], "candidate_scores": []},
]
plan = plan_screening(projects, opportunities)
rows = build_screen_delta(spark, projects, opportunities, matched, plan).collect()
observed = sorted((row["project_id"], row["opportunity_id"], row["screen_reason"],
                   row["candidate_score"]) for row in rows)
expected = [(1, "s1", "candidate", 100.0), (1, "s2", "country_conflict", None),
            (2, "s1", "country_conflict", None), (2, "s2", "country_conflict", None)]
if observed != expected:
    raise AssertionError(f"baseline screening mismatch: {observed}")

new_projects = [*projects, {"project_id": 3, "project_name": "Desert Hydrogen", "country": ""}]
new_opportunities = [*opportunities, {"opportunity_id": "s3", "opportunity_name": "North LNG Terminal",
                                     "country": "Brazil"}]
new_matches = [
    {"project_id": 1, "candidate_ids": ["s1", "s3"], "candidate_scores": [100.0, 100.0]},
    {"project_id": 2, "candidate_ids": [], "candidate_scores": []},
    {"project_id": 3, "candidate_ids": [], "candidate_scores": []},
]
delta_plan = plan_screening(new_projects, new_opportunities, projects, opportunities, SCREENING_VERSION)
delta_rows = build_screen_delta(spark, new_projects, new_opportunities, new_matches, delta_plan).collect()
delta_observed = sorted((row["project_id"], row["opportunity_id"], row["screen_reason"],
                         row["candidate_score"]) for row in delta_rows)
delta_expected = [(1, "s3", "candidate", 100.0), (2, "s3", "country_conflict", None),
                  (3, "s1", "no_name_overlap", None), (3, "s2", "no_name_overlap", None),
                  (3, "s3", "no_name_overlap", None)]
if delta_observed != delta_expected or delta_plan["screened_pairs"] != 5 or delta_plan["possible_pairs"] != 9:
    raise AssertionError(f"two-sided delta mismatch: {delta_observed}")
dbutils.notebook.exit(json.dumps({"status": "passed", "baseline_pairs": len(rows),
                                  "delta_pairs": len(delta_rows), "possible_pairs": delta_plan["possible_pairs"]}))